<a href="https://colab.research.google.com/github/yulimmm/web-crawling/blob/getReview/GetReviews_iOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install xmltodict

In [2]:
pip install pandas

In [3]:
pip install requests

In [5]:
import requests
import xmltodict
import pandas as pd
import os
import time

# 텍스트만 추출하는 함수
def extract_text_content(content):
    if isinstance(content, list):
        for item in content:
            if item.get('@type') == 'text':
                return item.get('#text', '').strip()
    elif isinstance(content, dict):
        return content.get('#text', '').strip()
    return str(content).strip()

# 여러 페이지에서 리뷰 수집
def get_ios_reviews_all(app_id, country, max_page=10):
    reviews = []

    for page in range(1, max_page + 1):
        url = f'https://itunes.apple.com/{country}/rss/customerreviews/page={page}/id={app_id}/sortBy=mostRecent/xml'
        response = requests.get(url)
        if response.status_code != 200:
            print(f"⚠️ 페이지 {page} 요청 실패 (status code: {response.status_code})")
            break

        xml_str = response.content.decode('utf-8')
        try:
            data = xmltodict.parse(xml_str)
        except Exception as e:
            print(f"⚠️ XML 파싱 실패 (page {page}): {e}")
            break

        entries = data['feed'].get('entry', [])
        if isinstance(entries, dict):
            entries = [entries]

        if len(entries) <= 1:  # 리뷰가 없거나 앱 정보만 있을 경우 종료
            break

        for entry in entries:
            if 'author' not in entry:
                continue
            reviews.append({
                '작성자': entry['author']['name'],
                '별점': int(entry['im:rating']),
                '제목': entry['title'],
                '내용': extract_text_content(entry['content']),
                '작성일': entry['updated'],
                '앱버전': entry.get('im:version', '')
            })

        time.sleep(0.5)  # 서버에 무리 안 가게 대기

    # DataFrame 생성 및 저장
    df = pd.DataFrame(reviews)
    df['작성일'] = pd.to_datetime(df['작성일'])

    csv_filename = f'app_reviews_{app_id}_{country}.csv'
    df.to_csv(csv_filename, index=False, encoding='utf-8-sig')

    print(f"✅ 총 {len(df)}개 리뷰 저장 완료: {os.path.abspath(csv_filename)}")
    return df

# 앱 ID와 국가 설정
country = 'kr'
ios_app_id = '1601173965'

# 실행 (최대 10페이지)
df_reviews = get_ios_reviews_all(ios_app_id, country, max_page=10)
print(df_reviews.head())


✅ 총 27개 리뷰 저장 완료: /content/app_reviews_1601173965_kr.csv
             작성자  별점          제목  \
0         피파의 마덜   1   버그가 좀 있네요   
1            짱소방   1  업데이트 후 최악임   
2          ㅋㅋ키ㅣㅣ   1  전 버전으로 돌려요   
3        hyyeonn   1  원래대로 돌려주세요   
4  dingddingding   1     업데이트 문제   

                                                  내용  \
0  팟을 등록까지 했는데 폰을 껐다 키면 \n사커비 앱이 로그아웃되면서 팟이 이미 다른...   
1  업데이트 전으로 돌려줘요 진짜 개불편하고 시간입력하는거는 전에도 불편했는데 지금른 ...   
2     최악임. 객관적으로 기록도 안돼고 오류 엄청\n많이 나오 이럴거면 업데이트\n왜함?   
3  기존 UI 대비 정말 심각한 수준입니다.\n어떤 업데이트를 바라고 하신지 모르겠지만...   
4                                업데이트 하고 계속 연결 끊겨요..   

                        작성일     앱버전  
0 2025-05-20 04:05:02-07:00  5.0.14  
1 2025-02-26 00:17:50-07:00  5.0.12  
2 2025-02-23 00:41:23-07:00    5.11  
3 2025-02-01 09:11:22-07:00    5.10  
4 2025-01-17 19:50:14-07:00    5.10  


In [6]:
pip install google_play_scraper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.4 MB/s eta 0:00:00


In [7]:
from google_play_scraper import reviews, Sort
import pandas as pd
import os

def get_android_reviews_all(app_id, lang='ko', country='kr', max_reviews=500):
    all_reviews = []
    count_per_call = 100
    token = None

    while len(all_reviews) < max_reviews:
        n_to_fetch = min(count_per_call, max_reviews - len(all_reviews))
        result, token = reviews(
            app_id,
            lang=lang,
            country=country,
            sort=Sort.NEWEST,
            count=n_to_fetch,
            continuation_token=token
        )

        if not result:
            break

        for r in result:
            all_reviews.append({
                '작성자': r['userName'],
                '별점': r['score'],
                '제목': r.get('reviewCreatedVersion', ''),
                '내용': r['content'],
                '작성일': r['at'],
                '앱버전': r.get('reviewCreatedVersion', '')
            })

    df = pd.DataFrame(all_reviews)
    df['작성일'] = pd.to_datetime(df['작성일'])

    csv_filename = f'android_reviews_{app_id}.csv'
    df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
    print(f"✅ 총 {len(df)}개 리뷰 저장 완료: {os.path.abspath(csv_filename)}")

    return df

# 앱 ID 예시: 카카오톡 (com.kakao.talk)
android_app_id = 'com.ubeeslab.mybee'

df_android = get_android_reviews_all(android_app_id, max_reviews=300)
print(df_android.head())


✅ 총 43개 리뷰 저장 완료: /content/android_reviews_com.ubeeslab.mybee.csv
            작성자  별점      제목  \
0    Johnny Lee   1  5.0.10   
1           강인승   1  5.0.10   
2          HS J   5  5.0.10   
3          Ty K   1     3.4   
4  Youngchul Ko   1  5.0.10   

                                                  내용                 작성일  \
0              앱 업데이트 이후로, 운동시간, 뛴거리 등 정보가 너무 부정확해요. 2025-02-09 00:55:44   
1  리뷰 하기전에 궁금한거 Q. 싸커비 1세대(현재 품절)는 앱과 연동이 안되는겁니까?... 2025-02-07 14:59:41   
2  업데이트 이후 제공해 주는 데이터가 세분화되서 앞으로 더 잘 쓸 것 같음 데이터가 ... 2025-02-07 09:36:45   
3  소보원에 고발하기전에 연락바람 앱작동 안된다는 리뷰가 수십개인데 장비 팔아먹고 이용... 2025-01-26 08:30:44   
4  업데이트되고 정보입력이 너무 불편합니다ㅡㅡ시간그래프보고 입력해도 실제하고 안맞구요ㅡ... 2025-01-19 14:08:29   

      앱버전  
0  5.0.10  
1  5.0.10  
2  5.0.10  
3     3.4  
4  5.0.10  
